# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [1]:
def lettre_vers_nombre(c):
    return ord(c) - ord("A")

def nombre_vers_lettre(n):
    return chr(n + ord("A"))

print(lettre_vers_nombre("C"))
print(nombre_vers_lettre(2))

2
C


## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [2]:
import unicodedata

def nettoyer(texte):
    texte = unicodedata.normalize("NFD", texte).upper()
    resultat = ""
    for c in texte:
        if "A" <= c <= "Z":
            resultat = resultat + c
    return resultat

print(nettoyer("Été, à Paris !"))

ETEAPARIS


## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [3]:
def chiffrer(texte, a, b):
    resultat = ""
    for c in texte:
        x = lettre_vers_nombre(c)
        y = (a * x + b) % 26
        resultat = resultat + nombre_vers_lettre(y)
    return resultat

print(chiffrer("ATTAQUEALAUBE", 5, 8))

IZZIKECILIENC


## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [4]:
import math

valides = []
for a in range(1, 26):
    if math.gcd(a, 26) == 1:
        valides.append(a)

print(valides)
print(len(valides))

[1, 3, 5, 7, 9, 11, 15, 17, 19, 21, 23, 25]
12


## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [5]:
def inverse_modulaire(a, n):
    for x in range(1, n):
        if (a * x) % n == 1:
            return x
    raise ValueError("pas d'inverse")

print(inverse_modulaire(5, 26))

21


## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [6]:
def dechiffrer(texte, a, b):
    if math.gcd(a, 26) != 1:
        raise ValueError("clé invalide")
    inverse = inverse_modulaire(a, 26)
    resultat = ""
    for c in texte:
        y = lettre_vers_nombre(c)
        x = (inverse * (y - b)) % 26
        resultat = resultat + nombre_vers_lettre(x)
    return resultat

m = "Attaque à l'aube"
print(nettoyer(m))
print(dechiffrer(chiffrer(nettoyer(m), 5, 8), 5, 8))

ATTAQUEALAUBE
ATTAQUEALAUBE


## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [7]:
secret = chiffrer(nettoyer("Le chiffrement affine est très facile à casser, car il existe seulement trois cent douze clés possibles à essayer."), 7, 3)
print(secret)

def toutes_les_clefs(secret):
    liste = []
    for a in valides:
        for b in range(26):
            liste.append((a, b, dechiffrer(secret, a, b)))
    return liste

propositions = toutes_les_clefs(secret)
print(len(propositions))
for a, b, texte in propositions:
    print(a, b, texte)

CFRAHMMSFJFQGDMMHQFFZGGSFZMDRHCFDRDZZFSRDSHCFIHZGFZFNCFJFQGGSXHZRFQGYXNWFRCFZEXZZHKCFZDFZZDPFS
312
1 0 CFRAHMMSFJFQGDMMHQFFZGGSFZMDRHCFDRDZZFSRDSHCFIHZGFZFNCFJFQGGSXHZRFQGYXNWFRCFZEXZZHKCFZDFZZDPFS
1 1 BEQZGLLREIEPFCLLGPEEYFFREYLCQGBECQCYYERQCRGBEHGYFEYEMBEIEPFFRWGYQEPFXWMVEQBEYDWYYGJBEYCEYYCOER
1 2 ADPYFKKQDHDOEBKKFODDXEEQDXKBPFADBPBXXDQPBQFADGFXEDXDLADHDOEEQVFXPDOEWVLUDPADXCVXXFIADXBDXXBNDQ
1 3 ZCOXEJJPCGCNDAJJENCCWDDPCWJAOEZCAOAWWCPOAPEZCFEWDCWCKZCGCNDDPUEWOCNDVUKTCOZCWBUWWEHZCWACWWAMCP
1 4 YBNWDIIOBFBMCZIIDMBBVCCOBVIZNDYBZNZVVBONZODYBEDVCBVBJYBFBMCCOTDVNBMCUTJSBNYBVATVVDGYBVZBVVZLBO
1 5 XAMVCHHNAEALBYHHCLAAUBBNAUHYMCXAYMYUUANMYNCXADCUBAUAIXAEALBBNSCUMALBTSIRAMXAUZSUUCFXAUYAUUYKAN
1 6 WZLUBGGMZDZKAXGGBKZZTAAMZTGXLBWZXLXTTZMLXMBWZCBTAZTZHWZDZKAAMRBTLZKASRHQZLWZTYRTTBEWZTXZTTXJZM
1 7 VYKTAFFLYCYJZWFFAJYYSZZLYSFWKAVYWKWSSYLKWLAVYBASZYSYGVYCYJZZLQASKYJZRQGPYKVYSXQSSADVYSWYSSWIYL
1 8 UXJSZEEKXBXIYVEEZIXXRYYKXREVJZUXVJVRRXKJVKZUXAZRYXRXFUXBXIYYKPZRJXIYQPFOXJUXRWPRRZCUXRVXRRVHXK
1 9 TWIRYD

## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [8]:
compte = {}
for c in secret:
    compte[c] = compte.get(c, 0) + 1

plus_frequente = "A"
for c in compte:
    if compte[c] > compte[plus_frequente]:
        plus_frequente = c
print(plus_frequente, compte[plus_frequente])

f = lettre_vers_nombre(plus_frequente)
candidats = []
for a in valides:
    b = (f - 4 * a) % 26
    candidats.append((a, b, dechiffrer(secret, a, b)))

for a, b, texte in candidats:
    print(a, b, texte)

meilleur = candidats[0]
meilleur_score = -1
for candidat in candidats:
    score = 0
    for mot in ["LE", "LA", "DE", "ET", "EST", "UN"]:
        score = score + candidat[2].count(mot)
    if score > meilleur_score:
        meilleur_score = score
        meilleur = candidat
print(meilleur)

F 19
1 1 BEQZGLLREIEPFCLLGPEEYFFREYLCQGBECQCYYERQCRGBEHGYFEYEMBEIEPFFRWGYQEPFXWMVEQBEYDWYYGJBEYCEYYCOER
3 19 DEILWPPREOEZNMPPWZEECNNRECPMIWDEMIMCCERIMRWDEFWCNECEYDEOEZNNRKWCIEZNTKYBEIDECVKCCWXDECMECCMQER
5 11 TEWDUVVREKEBZOVVUBEEIZZREIVOWUTEOWOIIERWORUTEPUIZEIEQTEKEBZZRSUIWEBZNSQXEWTEIJSIIUFTEIOEIIOGER
7 3 LECHIFFREMENTAFFINEESTTRESFACILEACASSERCARILEXISTESEULEMENTTROISCENTDOUZECLESPOSSIBLESAESSAYER
9 21 VEOPKZZREQELHYZZKLEEMHHREMZYOKVEYOYMMEROYRKVENKMHEMECVEQELHHRGKMOELHJGCDEOVEMBGMMKTVEMYEMMYIER
11 13 ZEYNQHHRECEFXSHHQFEEUXXREUHSYQZESYSUUERYSRQZEJQUXEUEAZECEFXXRIQUYEFXBIAPEYZEULIUUQVZEUSEUUSMER
15 23 JEKVSBBREGEDLQBBSDEEOLLREOBQKSJEQKQOOERKQRSJEZSOLEOEIJEGEDLLRASOKEDLHAITEKJEOXAOOSNJEOQEOOQWER
17 15 NEUTYJJRESEXBKJJYXEEWBBREWJKUYNEKUKWWERUKRYNEVYWBEWEGNESEXBBRCYWUEXBZCGFEUNEWHCWWYPNEWKEWWKAER
19 7 XEGBADDREWEVPIDDAVEEQPPREQDIGAXEIGIQQERGIRAXELAQPEQEOXEWEVPPRUAQGEVPFUOJEGXEQTUQQAHXEQIEQQIKER
21 25 PEMFONNREYEHJUNNOHEEAJJREANUMOPEUMUAAERMUROPETOAJEAESPEYEHJJRQOAMEHJVQSLEMPEAZQAAODPEAUE

## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [9]:
import random

texte_long = nettoyer("La cryptographie est l'art de protéger un message en le rendant illisible pour tous ceux qui ne possèdent pas la clé. Depuis l'Antiquité, les hommes ont cherché à cacher leurs secrets, et chaque nouvelle méthode de chiffrement a fini par susciter une nouvelle méthode pour la casser. Le chiffrement affine, que nous avons étudié, remplace chaque lettre par une autre selon une formule simple. Il paraît solide, mais il ne l'est pas du tout, car le nombre de clés possibles est très petit et car les lettres les plus fréquentes du français restent les plus fréquentes dans le message chiffré. Un attaquant patient n'a qu'à compter les lettres pour deviner la clé. Pour résister, il faut mélanger davantage les lettres, par exemple en utilisant plusieurs décalages successifs, comme le fait le chiffre de Vigenère. Les cryptographes modernes utilisent des clés immenses et des calculs que même les meilleurs ordinateurs ne peuvent pas refaire en un temps raisonnable. Cette sécurité repose sur les mathématiques, et en particulier sur l'arithmétique des nombres premiers.")
secret_long = chiffrer(texte_long, 11, 5)

def indice_coincidence(texte):
    n = len(texte)
    total = 0
    for lettre in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        k = texte.count(lettre)
        total = total + k * (k - 1)
    return total / (n * (n - 1))

hasard = ""
for i in range(len(texte_long)):
    hasard = hasard + random.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

print(indice_coincidence(texte_long))
print(indice_coincidence(secret_long))
print(indice_coincidence(hasard))

0.08190650539391958
0.08190650539391958
0.03890944753187316


## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [10]:
def vigenere(texte, cle):
    resultat = ""
    for i in range(len(texte)):
        x = lettre_vers_nombre(texte[i])
        decalage = lettre_vers_nombre(cle[i % len(cle)])
        resultat = resultat + nombre_vers_lettre((x + decalage) % 26)
    return resultat

chiffre_vigenere = vigenere(texte_long, "CLE")
print(chiffre_vigenere[:40])
print(indice_coincidence(chiffre_vigenere))

NLGTJTVZKTLTJTIGDXNLVVOIRCSVPKGCYPXIUDEI
0.049731284733573064


## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [11]:
for L in range(1, 8):
    total = 0
    for i in range(L):
        colonne = chiffre_vigenere[i::L]
        total = total + indice_coincidence(colonne)
    print(L, round(total / L, 4))

1 0.0497
2 0.0501
3 0.0831
4 0.0505
5 0.0499
6 0.0843
7 0.0483
